# Vue3 框架 · 第 13 天

## 大事件后台管理系统项目实战

> 本篇依据“大事件管理系统”项目的 MD 笔记、PPT 和 Vue3 项目代码整理。项目使用 Vue 3、Vite、Vue Router 4、Pinia、Element Plus、Axios、ESLint/Prettier 和 Husky，完成登录、布局、文章分类、文章管理和个人中心。

这一天的重点是把前面学到的组合式 API、Pinia、路由和组件封装真正落到一个后台项目中。

## 一、项目功能与技术栈

### 主要页面

- 登录与注册：表单校验、验证码提示、token 登录。
- 首页布局：侧边菜单、顶部用户信息、退出登录。
- 文章分类：表格展示、添加、编辑、删除。
- 文章管理：分类筛选、搜索、分页、发布、编辑、删除。
- 个人中心：基本资料、头像上传、密码修改。

### 技术栈

```text
Vue 3 + <script setup>
Vite + pnpm
Vue Router 4
Pinia + 持久化
Element Plus
Axios
ESLint + Prettier + Husky
```

项目的组织原则：页面组件放 `views`，通用组件放 `components`，请求放 `api`，状态放 `stores`，Axios 实例放 `utils/request.js`。

## 二、pnpm 创建项目与工程规范

```bash
pnpm create vue
cd big-event-admin
pnpm install
pnpm dev
```

### 推荐目录

```text
src/
├─ api/                 接口模块
├─ assets/              图片和全局样式
├─ components/          可复用组件
├─ router/              路由
├─ stores/              Pinia 仓库
├─ utils/               request 等工具
├─ views/               页面组件
│  ├─ login/
│  ├─ layout/
│  ├─ article/
│  └─ user/
├─ App.vue
└─ main.js
```

### ESLint、Prettier、Husky

- ESLint 负责发现语法和代码质量问题。
- Prettier 统一格式，避免团队提交产生无意义的格式差异。
- Husky 在提交前执行检查，防止明显错误进入仓库。

```bash
pnpm lint
pnpm format
```

不要为了通过检查直接关闭规则；先修复未使用变量、组件命名、缩进和类型错误。

## 三、Vue Router 4 路由设计

### 3.1 创建路由

```js
// src/router/index.js
import { createRouter, createWebHistory } from 'vue-router'
import LoginPage from '@/views/login/LoginPage.vue'
import LayoutContainer from '@/views/layout/LayoutContainer.vue'

const router = createRouter({
  history: createWebHistory(),
  routes: [
    { path: '/login', component: LoginPage },
    {
      path: '/',
      component: LayoutContainer,
      redirect: '/article/manage',
      children: [
        { path: '/article/manage', component: () => import('@/views/article/ArticleManage.vue') },
        { path: '/article/channel', component: () => import('@/views/article/ArticleChannel.vue') },
        { path: '/user/profile', component: () => import('@/views/user/UserProfile.vue') }
      ]
    }
  ]
})

export default router
```

页面路由使用懒加载，公共后台外壳只加载一次。布局组件中必须放 `<router-view />`，子页面才有显示位置。

## 四、Pinia 用户仓库与持久化

```js
// src/stores/modules/user.js
import { ref } from 'vue'
import { defineStore } from 'pinia'

export const useUserStore = defineStore('user', () => {
  const token = ref(localStorage.getItem('token') || '')
  const userInfo = ref(null)

  const setToken = value => {
    token.value = value
    value ? localStorage.setItem('token', value) : localStorage.removeItem('token')
  }
  const clearUser = () => {
    setToken('')
    userInfo.value = null
  }

  return { token, userInfo, setToken, clearUser }
}, { persist: true })
```

登录成功后只通过 store 的 action 或方法更新 token；退出时清空 token、用户信息和本地持久化数据。任何页面判断登录状态都读取同一份 store，避免各页面自己维护 token。

## 五、Axios 请求工具设计

```js
// src/utils/request.js
import axios from 'axios'
import { ElMessage } from 'element-plus'
import { useUserStore } from '@/stores/modules/user'

const request = axios.create({
  baseURL: import.meta.env.VITE_API_BASE_URL,
  timeout: 10000
})

request.interceptors.request.use(config => {
  const userStore = useUserStore()
  if (userStore.token) config.headers.Authorization = userStore.token
  return config
})

request.interceptors.response.use(
  response => response.data,
  error => {
    ElMessage.error(error.response?.data?.message || '请求失败')
    return Promise.reject(error)
  }
)

export default request
```

接口按业务拆分：

```js
// src/api/user.js
import request from '@/utils/request'
export const userRegisterService = data => request.post('/api/reg', data)
export const userLoginService = data => request.post('/api/login', data)
export const userGetInfoService = () => request.get('/my/userinfo')
export const userUpdateInfoService = data => request.put('/my/userinfo', data)
```

页面只组合参数和处理结果，baseURL、token、错误提示等公共逻辑全部由请求工具处理。

## 六、登录与注册页面

### 6.1 表单和校验

```vue
<template>
  <el-form ref="formRef" :model="form" :rules="rules">
    <el-form-item prop="username">
      <el-input v-model="form.username" placeholder="请输入用户名" />
    </el-form-item>
    <el-form-item prop="password">
      <el-input v-model="form.password" type="password" show-password />
    </el-form-item>
    <el-button type="primary" @click="submit">登录</el-button>
  </el-form>
</template>

<script setup>
import { reactive, ref } from 'vue'
const formRef = ref()
const form = reactive({ username: '', password: '' })
const rules = {
  username: [{ required: true, message: '请输入用户名', trigger: 'blur' }],
  password: [{ required: true, message: '请输入密码', trigger: 'blur' }]
}
</script>
```

提交前先调用 `formRef.value.validate()`，校验通过后再请求。注册页还要校验确认密码、用户名长度和密码格式。

### 6.2 登录成功

```js
const submit = async () => {
  await formRef.value.validate()
  const res = await userLoginService(form)
  userStore.setToken(res.token)
  await router.replace(route.query.redirect || '/')
}
```

登录页可以通过查询参数保存原访问地址，让用户登录后回到原页面。

## 七、路由前置守卫与后台 Layout

```js
router.beforeEach((to) => {
  const userStore = useUserStore()
  if (to.path !== '/login' && !userStore.token) {
    return { path: '/login', query: { redirect: to.fullPath } }
  }
  if (to.path === '/login' && userStore.token) {
    return '/'
  }
})
```

`LayoutContainer.vue` 负责：

- Element Plus 菜单和路由导航。
- 顶部面包屑、用户头像和退出按钮。
- `<router-view />` 显示子页面。

```vue
<el-menu router :default-active="$route.path">
  <el-menu-item index="/article/manage">文章管理</el-menu-item>
  <el-menu-item index="/article/channel">文章分类</el-menu-item>
  <el-menu-item index="/user/profile">个人中心</el-menu-item>
</el-menu>
```

退出时弹出确认框，调用 `userStore.clearUser()` 后跳转到登录页。

## 八、文章分类管理

### 8.1 表格渲染

```js
// api/article.js
export const getChannelService = () => request.get('/my/cate')
export const addChannelService = data => request.post('/my/cate', data)
export const updateChannelService = data => request.put('/my/cate', data)
export const deleteChannelService = id => request.delete('/my/cate', { params: { id } })
```

```vue
<el-table v-loading="loading" :data="list">
  <el-table-column prop="id" label="序号" />
  <el-table-column prop="cate_name" label="分类名称" />
  <el-table-column prop="cate_alias" label="分类别名" />
  <el-table-column label="操作">
    <template #default="{ row }">
      <el-button link @click="edit(row)">编辑</el-button>
      <el-button link type="danger" @click="remove(row)">删除</el-button>
    </template>
  </el-table-column>
</el-table>
```

请求期间显示 `v-loading`，成功后更新列表，失败由 Axios 拦截器提示。

### 8.2 添加、编辑和删除

添加和编辑可以复用 `ChannelEdit.vue` 弹层组件，通过 `v-model:visible` 控制显示：

```vue
<ChannelEdit
  v-model:visible="dialogVisible"
  :channel="editingChannel"
  @success="getChannelList"
/>
```

弹层内部提交：

```js
await formRef.value.validate()
if (props.channel.id) {
  await updateChannelService(form)
} else {
  await addChannelService(form)
}
ElMessage.success('操作成功')
emits('success')
```

删除前使用 `ElMessageBox.confirm`，删除成功后重新获取列表，避免本地列表和服务端状态不一致。

## 九、文章管理页面

### 9.1 筛选、分页和重置

```vue
<el-form :model="params" inline>
  <el-form-item label="文章分类">
    <el-select v-model="params.cate_id" clearable>
      <el-option v-for="item in channels" :key="item.id" :label="item.cate_name" :value="item.id" />
    </el-select>
  </el-form-item>
  <el-form-item label="文章状态">
    <el-select v-model="params.state" clearable>
      <el-option label="已发布" value="已发布" />
      <el-option label="草稿" value="草稿" />
    </el-select>
  </el-form-item>
  <el-button type="primary" @click="search">搜索</el-button>
  <el-button @click="reset">重置</el-button>
</el-form>

<el-pagination
  v-model:current-page="params.page"
  v-model:page-size="params.pageSize"
  :total="total"
  @change="getArticleList"
/>
```

搜索时将页码重置为 1；重置时恢复默认参数并重新请求。列表请求开启 loading，分页数据和总数都从接口响应中读取。

### 9.2 国际化和状态展示

文章状态、按钮文案等可以通过统一映射处理。不要在模板中散落大量三元表达式；把展示文字和样式整理为计算属性或方法。

## 十、文章发布、编辑与删除

文章编辑适合使用 Element Plus 抽屉组件：

```vue
<el-drawer v-model="drawerVisible" title="文章编辑" size="50%">
  <ArticleEdit
    :article="editingArticle"
    @success="handleSuccess"
  />
</el-drawer>
```

文章表单包括标题、分类、封面、内容和状态：

- 用 `el-upload` 上传图片，拿到服务端返回的 URL。
- 用 `vue-quill` 编辑富文本内容。
- 新增和编辑共用一个表单，编辑时先回显数据。
- 提交成功后清空表单、关闭抽屉并重新拉取列表。

```js
const submit = async () => {
  await formRef.value.validate()
  const service = form.id ? updateArticleService : addArticleService
  await service(form)
  ElMessage.success('保存成功')
  emit('success')
}
```

删除文章前必须二次确认，并在删除成功后更新分页列表；如果删除最后一页的最后一条，需要处理页码回退。

## 十一、个人中心

### 11.1 基本资料

```vue
<el-form ref="formRef" :model="form" :rules="rules">
  <el-form-item label="用户名"><el-input v-model="form.username" disabled /></el-form-item>
  <el-form-item label="昵称" prop="nickname"><el-input v-model="form.nickname" /></el-form-item>
  <el-form-item label="邮箱" prop="email"><el-input v-model="form.email" /></el-form-item>
  <el-button type="primary" @click="save">保存</el-button>
</el-form>
```

获取资料后更新表单，提交前校验邮箱格式；成功后同步更新 Pinia 中的用户信息，顶部头像和昵称可以立即刷新。

### 11.2 头像上传

选择图片后先预览，再提交：

```js
const onSelectFile = file => {
  avatarUrl.value = URL.createObjectURL(file.raw)
}
const uploadAvatar = async () => {
  const formData = new FormData()
  formData.append('avatar', file.value)
  await userUploadAvatarService(formData)
}
```

上传成功后释放旧的 `ObjectURL`，并更新用户仓库中的头像地址。

### 11.3 重置密码

校验原密码、新密码和确认密码一致后调用更新接口。成功后清空表单并提示用户重新登录；不要把密码保存在 localStorage。

## 十二、AI 工具辅助开发

项目资料还介绍了 ChatGPT Prompt 和 GitHub Copilot 的使用。正确做法是把工具用于：

- 解释报错、补充测试数据和生成重复模板。
- 根据明确的接口字段生成表格列和校验规则。
- 让 Copilot 生成初稿后，人工核对权限、接口参数、错误处理和安全性。

高质量 Prompt 应包含目标、上下文、输入输出格式、限制条件和验收标准。例如：

```text
请为 Vue 3 + Element Plus 的文章分类表单生成校验规则。
字段：cate_name 必填且 2~10 个字符，cate_alias 必填且只能包含字母数字下划线。
输出：只返回 rules 对象，并说明每条规则的触发时机。
```

工具生成的代码必须结合项目现有 API 和命名约定检查，不能直接覆盖真实业务逻辑。

## 十三、项目验收与易错点

- 登录后刷新页面仍能保留 token，未登录不能进入后台。
- Axios 请求自动携带 token，401 时清理状态并跳回登录。
- 页面组件使用路由懒加载，Layout 保持公共菜单和出口。
- 表格请求有 loading、空数据和错误提示。
- 弹层新增、编辑共用表单，提交后重新拉取数据。
- 文件上传使用 `FormData`，富文本内容和图片 URL 经过接口确认。
- 退出登录清理 Pinia 和本地持久化，防止账号数据串用。
- 生产打包前执行 lint，检查环境变量和接口 baseURL。

常见问题：

1. `useUserStore()` 必须在 Pinia 安装后或正确的 setup 上下文中调用。
2. `v-model` 绑定 Element Plus 组件时确认属性类型，分页 `current-page` 和 `page-size` 要与接口参数同步。
3. 弹层编辑时要复制数据，避免用户取消编辑却修改了列表原对象。
4. 删除最后一页数据后要修正当前页，否则会出现空页面。

## 十四、当天速记

- 后台项目先搭工程规范和目录，再搭路由、请求工具和用户状态。
- Pinia 统一管理 token、用户信息和持久化；Axios 拦截器统一处理请求头和错误。
- Element Plus 表单负责输入校验，表格负责列表渲染，弹层和抽屉负责新增编辑。
- 文章分类是 CRUD 基础案例，文章管理把筛选、分页、上传和富文本组合起来。
- 个人中心要同步修改用户仓库，退出时清空所有用户相关状态。
- AI 工具可以提高重复代码的产出速度，但接口、权限、错误处理和数据安全必须人工核对。